# 1. Data Loading and Initial Inspection

The dataset is loaded and inspected to understand its structure, data types, and missing values. 
A copy of the dataset is created to preserve the original raw data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
# Load the raw dataset
df = pd.read_csv("data/raw/FullData.csv")

# Keep only required columns
df = df[
    [
        "Name",
        "Nationality",
        "National_Position",
        "Club",
        "Club_Position",
        "Rating",
        "Height",
        "Weight",
        "Birth_Date",
        "Age",
        "Skill_Moves",
        "Ball_Control",
        "Dribbling",
        "Speed",
        "Strength"
    ]
]

# Preview
df.head()

,Name,Nationality,National_Position,Club,Club_Position,Rating,Height,Weight,Birth_Date,Age,Skill_Moves,Ball_Control,Dribbling,Speed,Strength
0,Cristiano Ronaldo,Portugal,LS,Real Madrid,LW,94,185 cm,80 kg,02/05/1985,32,5,93,92,92,80
1,Lionel Messi,Argentina,RW,FC Barcelona,RW,93,170 cm,72 kg,06/24/1987,29,4,95,97,87,59
2,Neymar,Brazil,LW,FC Barcelona,LW,92,174 cm,68 kg,02/05/1992,25,5,95,96,90,49
3,Luis Suárez,Uruguay,LS,FC Barcelona,ST,92,182 cm,85 kg,01/24/1987,30,4,91,86,77,76
4,Manuel Neuer,Germany,GK,FC Bayern,GK,92,193 cm,92 kg,03/27/1986,31,1,48,30,61,83


In [3]:
# Check the shape of the dataset
df.shape

(17588, 15)

In [4]:
# Check data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17588 entries, 0 to 17587
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Name               17588 non-null  str  
 1   Nationality        17588 non-null  str  
 2   National_Position  1075 non-null   str  
 3   Club               17588 non-null  str  
 4   Club_Position      17587 non-null  str  
 5   Rating             17588 non-null  int64
 6   Height             17588 non-null  str  
 7   Weight             17588 non-null  str  
 8   Birth_Date         17588 non-null  str  
 9   Age                17588 non-null  int64
 10  Skill_Moves        17588 non-null  int64
 11  Ball_Control       17588 non-null  int64
 12  Dribbling          17588 non-null  int64
 13  Speed              17588 non-null  int64
 14  Strength           17588 non-null  int64
dtypes: int64(7), str(8)
memory usage: 2.0 MB


In [5]:
# Statistical summary of numerical columns
df.describe()

,Rating,Age,Skill_Moves,Ball_Control,Dribbling,Speed,Strength
count,17588.000000,17588.000000,17588.000000,17588.000000,17588.000000,17588.000000,17588.000000
mean,66.166193,25.460314,2.303161,57.972766,54.802877,65.483853,65.085854
std,7.083012,4.680217,0.746156,16.834779,18.913857,14.100615,12.532989
min,45.000000,17.000000,1.000000,5.000000,4.000000,11.000000,20.000000
25%,62.000000,22.000000,2.000000,53.000000,47.000000,58.000000,57.000000
50%,66.000000,25.000000,2.000000,63.000000,60.000000,68.000000,66.000000
75%,71.000000,29.000000,3.000000,69.000000,68.000000,75.000000,74.000000
max,94.000000,47.000000,5.000000,95.000000,97.000000,96.000000,98.000000


In [6]:
# Check for missing values
df.isnull().sum()

Name                     0
Nationality              0
National_Position    16513
Club                     0
Club_Position            1
Rating                   0
Height                   0
Weight                   0
Birth_Date               0
Age                      0
Skill_Moves              0
Ball_Control             0
Dribbling                0
Speed                    0
Strength                 0
dtype: int64

In [7]:
# Series example
example_series = pd.Series([1, 2, 3])
example_series

0    1
1    2
2    3
dtype: int64

In [8]:
# DataFrame example
example_df = pd.DataFrame({
    "A": [1, 2],
    "B": [3, 4]
})
example_df

,A,B
0,1,3
1,2,4


In [9]:
# iloc example
df.iloc[0:5, 0:5]

,Name,Nationality,National_Position,Club,Club_Position
0,Cristiano Ronaldo,Portugal,LS,Real Madrid,LW
1,Lionel Messi,Argentina,RW,FC Barcelona,RW
2,Neymar,Brazil,LW,FC Barcelona,LW
3,Luis Suárez,Uruguay,LS,FC Barcelona,ST
4,Manuel Neuer,Germany,GK,FC Bayern,GK


In [10]:
# boolean filtering
df[df["Rating"] >= 85].head()

,Name,Nationality,National_Position,Club,Club_Position,Rating,Height,Weight,Birth_Date,Age,Skill_Moves,Ball_Control,Dribbling,Speed,Strength
0,Cristiano Ronaldo,Portugal,LS,Real Madrid,LW,94,185 cm,80 kg,02/05/1985,32,5,93,92,92,80
1,Lionel Messi,Argentina,RW,FC Barcelona,RW,93,170 cm,72 kg,06/24/1987,29,4,95,97,87,59
2,Neymar,Brazil,LW,FC Barcelona,LW,92,174 cm,68 kg,02/05/1992,25,5,95,96,90,49
3,Luis Suárez,Uruguay,LS,FC Barcelona,ST,92,182 cm,85 kg,01/24/1987,30,4,91,86,77,76
4,Manuel Neuer,Germany,GK,FC Bayern,GK,92,193 cm,92 kg,03/27/1986,31,1,48,30,61,83


# 2. Cleaning and Transformation
The dataset is cleaned by removing duplicate records, handling missing values, converting data types, and transforming height and weight into more interpretable units. A new feature is also created to categorize player performance levels.

Duplicate names are not removed by name alone because the same player name may appear across different clubs. Instead, duplicates will be checked and removed using both Name and Club.

In [11]:
# Check true duplicates using Name + Club
df.duplicated(subset=["Name", "Club"]).sum()

np.int64(4)

In [12]:
# Remove true duplicate player-club records
df = df.drop_duplicates(subset=["Name", "Club"])

In [13]:
# Confirm duplicates were removed
df.duplicated(subset=["Name", "Club"]).sum()

np.int64(0)

In [34]:
# Fill missing national positions with "Unknown"
# These are not removed because many players do not have a national team position
df["National_Position"] = df["National_Position"].fillna("Unknown")

In [35]:
# Remove rows with missing club positions
df = df.dropna(subset=["Club_Position"])

In [15]:
# Convert birth date to datetime
df["Birth_Date"] = pd.to_datetime(df["Birth_Date"], errors="coerce")

In [16]:
# Convert selected columns to numeric
numeric_cols = [
    "Rating",
    "Age",
    "Skill_Moves",
    "Ball_Control",
    "Dribbling",
    "Speed",
    "Strength"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

Height and weight were originally recorded in centimeters and kilograms. They were converted to inches and pounds to make the data more interpretable.

In [36]:
# Clean Height: convert from strings like "185 cm" to inches
df["Height"] = df["Height"].astype(str).str.replace(" cm", "", regex=False).str.strip()
df["Height"] = pd.to_numeric(df["Height"], errors="coerce")
df["Height"] = (df["Height"] / 2.54).round(2)

In [37]:
# Clean Weight: convert from strings like "80 kg" to pounds
df["Weight"] = df["Weight"].astype(str).str.replace(" kg", "", regex=False).str.strip()
df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df["Weight"] = (df["Weight"] * 2.20462).round(2)

In [38]:
df[["Height", "Weight"]].head()

,Height,Weight
0,28.67,388.83
1,26.35,349.94
2,26.97,330.49
3,28.21,413.12
4,29.91,447.16


In [39]:
# Check missing values again after conversions
df.isnull().sum()

Name                 0
Nationality          0
National_Position    0
Club                 0
Club_Position        0
Rating               0
Height               0
Weight               0
Birth_Date           0
Age                  0
Skill_Moves          0
Ball_Control         0
Dribbling            0
Speed                0
Strength             0
dtype: int64

In [40]:
# Create a new column using np.where
df["Player_Level"] = np.where(
    df["Rating"] >= 80,
    "Elite",
    np.where(df["Rating"] >= 70, "Average", "Below Average")
)

In [41]:
# loc example
elite_players = df.loc[df["Player_Level"] == "Elite"]
elite_players.head()

,Name,Nationality,National_Position,Club,Club_Position,Rating,Height,Weight,Birth_Date,Age,Skill_Moves,Ball_Control,Dribbling,Speed,Strength,Player_Level
0,Cristiano Ronaldo,Portugal,LS,Real Madrid,LW,94,28.67,388.83,02/05/1985,32,5,93,92,92,80,Elite
1,Lionel Messi,Argentina,RW,FC Barcelona,RW,93,26.35,349.94,06/24/1987,29,4,95,97,87,59,Elite
2,Neymar,Brazil,LW,FC Barcelona,LW,92,26.97,330.49,02/05/1992,25,5,95,96,90,49,Elite
3,Luis Suárez,Uruguay,LS,FC Barcelona,ST,92,28.21,413.12,01/24/1987,30,4,91,86,77,76,Elite
4,Manuel Neuer,Germany,GK,FC Bayern,GK,92,29.91,447.16,03/27/1986,31,1,48,30,61,83,Elite


In [42]:
# Final check of data types and missing values
df.info()

<class 'pandas.DataFrame'>
Index: 17587 entries, 0 to 17587
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               17587 non-null  str    
 1   Nationality        17587 non-null  str    
 2   National_Position  17587 non-null  str    
 3   Club               17587 non-null  str    
 4   Club_Position      17587 non-null  str    
 5   Rating             17587 non-null  int64  
 6   Height             17587 non-null  float64
 7   Weight             17587 non-null  float64
 8   Birth_Date         17587 non-null  str    
 9   Age                17587 non-null  int64  
 10  Skill_Moves        17587 non-null  int64  
 11  Ball_Control       17587 non-null  int64  
 12  Dribbling          17587 non-null  int64  
 13  Speed              17587 non-null  int64  
 14  Strength           17587 non-null  int64  
 15  Player_Level       17587 non-null  str    
dtypes: float64(2), int64(7), str(7)
memory

In [43]:
df.isnull().sum()

Name                 0
Nationality          0
National_Position    0
Club                 0
Club_Position        0
Rating               0
Height               0
Weight               0
Birth_Date           0
Age                  0
Skill_Moves          0
Ball_Control         0
Dribbling            0
Speed                0
Strength             0
Player_Level         0
dtype: int64